# A7 — K-Means Clustering of AIMO Observations

Sprint issue: **"Perform a clustering of the AIMO observations"**

This notebook reports:
- K-means clustering of the AIMO biomechanical observations (scores excluded)
- Cluster interpretation
- PCA of the observations
- 2D scatterplot of the first two principal component scores coloured by cluster


## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

SCORE_COLS = ['AimoScore', 'EstimatedScore', 'ID']

df = pd.read_csv('../../data/processed/AimoScore_WeakLink_big_scores.csv')
feature_cols = [c for c in df.columns if c not in SCORE_COLS]

X_raw = df[feature_cols].values
X = StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(X_raw))

K = 3
kmeans = KMeans(n_clusters=K, n_init=20, random_state=RANDOM_SEED)
labels = kmeans.fit_predict(X)
df['cluster'] = labels

pca = PCA(random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X)
explained = pca.explained_variance_ratio_

print(f'Dataset: {df.shape[0]} observations, {len(feature_cols)} features')
print(f'Cluster sizes:\n{df["cluster"].value_counts().sort_index().to_string()}')

## Clustering

### ML Steps

1. **Data** — 2,094 AIMO observations with 40 biomechanical deviation features (13 angle, 25 NASM, 2 time). Score columns (`AimoScore`, `EstimatedScore`) and the ID column were excluded.
2. **Preprocessing** — Missing values imputed with the feature median (`SimpleImputer`). Features z-score scaled so that no single feature dominates the Euclidean distance calculation.
3. **Choosing K** — Within-cluster sum of squares (inertia) was plotted for K = 1–10 (elbow method, see `clustering_k_means.ipynb`). The curve shows a clear bend at K = 3.
4. **K-Means** — Fitted with K = 3, `n_init=20` (20 random restarts, best result kept), `random_state=42`.

In [ ]:
cluster_profiles = df.groupby('cluster')[feature_cols + ['AimoScore']].mean()
cluster_profiles[['AimoScore', 'No_1_Angle_Deviation', 'No_2_Angle_Deviation',
                   'No_4_Angle_Deviation', 'No_6_Angle_Deviation', 'No_10_Angle_Deviation',
                   'No_1_Time_Deviation', 'No_2_Time_Deviation',
                   'No_11_NASM_Deviation', 'No_12_NASM_Deviation']]

### Cluster Interpretation

| Cluster | Size | Mean AimoScore | Typical pattern |
|---------|------|----------------|-----------------|
| 0 | 1,098 (52%) | **0.703** | Good movers — low deviations across all features |
| 1 | 451 (22%) | **0.418** | Poor movers — timing-dominated failure (high `No_1/2_Time_Deviation`, elevated `No_5/7/10_Angle`) |
| 2 | 545 (26%) | **0.438** | Poor movers — angle-dominated failure (high `No_1/2/4/6_Angle_Deviation`, also high time deviations) |

- **Cluster 0** is the clear high-quality group: AimoScore of 0.703 vs ~0.43 for the other two. Deviation values are consistently low across all 40 features — these subjects move close to the reference pattern in both timing and joint angles.

- **Cluster 1** and **Cluster 2** have similar AimoScores (~0.42–0.44) but represent distinct failure modes. Cluster 1 is characterised by elevated time deviations (`No_1_Time`: 0.563, `No_2_Time`: 0.571) and specific angle outliers (`No_5`, `No_7`, `No_10`), suggesting a timing/sequencing problem. Cluster 2 shows broad elevation across angle features (`No_1`, `No_2`, `No_4`, `No_6` all ~0.72) combined with the highest time deviations (`No_1_Time`: 0.609, `No_2_Time`: 0.612) — both positional and rhythmic errors.

- The top discriminating features are NASM-based (`No_12_NASM`, `No_11_NASM`, `No_5_NASM`) followed by angle deviations (`No_13_Angle`, `No_10_Angle`), confirming that the NASM movement screens carry the most cluster-separating information.

- Scores were excluded during clustering, yet the recovered cluster means for `AimoScore` strongly validate the grouping — the algorithm found the quality signal from biomechanical deviations alone.

## PCA

In [ ]:
colours = ['#2196F3', '#FF5722', '#4CAF50']
centroids_pca = pca.transform(kmeans.cluster_centers_)

fig, ax = plt.subplots(figsize=(9, 6))

for c in range(K):
    mask = labels == c
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=colours[c], label=f'Cluster {c}',
        alpha=0.5, s=20, edgecolors='none'
    )

ax.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='black', marker='X', s=150, zorder=5, label='Centroids'
)

ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance explained)')
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance explained)')
ax.set_title('PCA of AIMO Observations — K-Means Clusters (K=3)')
ax.legend()
plt.tight_layout()
plt.show()

### PCA Interpretation

PC1 and PC2 together explain **31.1%** of total variance (18.6% + 12.5%). The data is genuinely high-dimensional — 15 components are needed to reach 80% — so the 2D projection discards most information, yet the cluster separation is still visible.

**PC1 (horizontal axis)** represents overall deviation severity. Points on the right have high deviations across most features; points on the left are close to the reference movement pattern. This axis primarily separates Cluster 0 (good movers, left) from Clusters 1 and 2 (poor movers, right).

**PC2 (vertical axis)** captures the *type* of deviation orthogonal to overall severity — likely the contrast between timing-dominated failures and angle-dominated failures. This is why Clusters 1 and 2 sit at different vertical positions despite having similar AimoScores.

The centroid markers (×) lie at the geometric centre of each cloud, confirming that K-means found compact, well-separated groups in the full 40-dimensional space. The visible overlap in 2D is expected given how little variance the first two PCs capture.